In [1]:
import pandas as pd
import tkinter as tk
from tkinter import filedialog, messagebox
import os

def separar_archivos_por_localidad():
    # Diccionario con los códigos y nombres de las localidades
    # (Nombres sin tildes para evitar problemas con las rutas de Windows)
    localidades = {
        "01": "Usaquen", "02": "Chapinero", "03": "Santa Fe", "04": "San Cristobal",
        "05": "Usme", "06": "Tunjuelito", "07": "Bosa", "08": "Kennedy",
        "09": "Fontibon", "10": "Engativa", "11": "Suba", "12": "Barrios Unidos",
        "13": "Teusaquillo", "14": "Los Martires", "15": "Antonio Narino",
        "16": "Puente Aranda", "17": "La Candelaria", "18": "Rafael Uribe Uribe",
        "19": "Ciudad Bolivar", "20": "Sumapaz"
    }

    # Ocultar la ventana principal de tkinter
    root = tk.Tk()
    root.withdraw()

    # 1. Solicitar el archivo de origen (la sábana completa)
    ruta_archivo = filedialog.askopenfilename(
        title="1. Selecciona el archivo Excel principal (Sábana de datos)",
        filetypes=[("Archivos Excel", "*.xlsx *.xls")]
    )
    if not ruta_archivo:
        print("Operación cancelada por el usuario.")
        return

    # 2. Solicitar la carpeta de destino
    carpeta_destino = filedialog.askdirectory(
        title="2. Selecciona la CARPETA DE DESTINO para guardar los reportes"
    )
    if not carpeta_destino:
        print("Operación cancelada por el usuario.")
        return

    try:
        print("Leyendo el archivo Excel... Esto puede tomar unos segundos.")
        df = pd.read_excel(ruta_archivo)

        # Validar que exista la columna de cruce
        if 'CODIGO_ENTIDAD' not in df.columns:
            messagebox.showerror("Error", "El archivo no contiene la columna 'CODIGO_ENTIDAD'.")
            return
        

        # Limpiar la columna: convertir a numérico, rellenar nulos, pasar a entero y formatear a 2 dígitos ('01', '02'...)
        df['CODIGO_TEMP'] = pd.to_numeric(df['CODIGO_ENTIDAD'], errors='coerce').fillna(-1).astype(int).astype(str).str.zfill(2)

        # Identificar las localidades únicas presentes en el archivo
        codigos_unicos = df['CODIGO_TEMP'].unique()
        archivos_creados = 0

        for codigo in codigos_unicos:
            # Ignorar filas donde el código de entidad estaba vacío o era texto inválido
            if codigo in ["-1", "-01"]:
                continue
            
            # Filtrar solo las filas correspondientes a esa localidad (y quitar la columna temporal)
            df_localidad = df[df['CODIGO_TEMP'] == codigo].drop(columns=['CODIGO_TEMP'])
            
            # Obtener el nombre de la localidad del diccionario (o marcar como Desconocida)
            nombre_loc = localidades.get(codigo, "Otra_Localidad")
            
            # Construir el nombre del archivo (Ej: 01_Usaquen.xlsx)
            nombre_archivo = f"{codigo}_{nombre_loc}.xlsx"
            ruta_salida = os.path.join(carpeta_destino, nombre_archivo)
            
            # Exportar a Excel
            df_localidad.to_excel(ruta_salida, index=False)
            archivos_creados += 1
            print(f"✅ Generado: {nombre_archivo} ({len(df_localidad)} registros)")

        # Notificación final
        messagebox.showinfo(
            "¡Proceso Terminado!", 
            f"Se crearon exitosamente {archivos_creados} archivos.\n\nRevisa la carpeta:\n{carpeta_destino}"
        )

    except Exception as e:
        messagebox.showerror("Error", f"Ocurrió un error al procesar el archivo:\n{str(e)}")

if __name__ == "__main__":
    separar_archivos_por_localidad()

Leyendo el archivo Excel... Esto puede tomar unos segundos.
✅ Generado: 04_San Cristobal.xlsx (3745 registros)
✅ Generado: 17_La Candelaria.xlsx (1295 registros)
✅ Generado: 14_Los Martires.xlsx (1523 registros)
✅ Generado: 19_Ciudad Bolivar.xlsx (3847 registros)
✅ Generado: 11_Suba.xlsx (2518 registros)
✅ Generado: 06_Tunjuelito.xlsx (1741 registros)
✅ Generado: 12_Barrios Unidos.xlsx (1450 registros)
✅ Generado: 08_Kennedy.xlsx (3758 registros)
✅ Generado: 05_Usme.xlsx (2847 registros)
✅ Generado: 02_Chapinero.xlsx (1511 registros)
✅ Generado: 15_Antonio Narino.xlsx (1324 registros)
✅ Generado: 07_Bosa.xlsx (3945 registros)
✅ Generado: 03_Santa Fe.xlsx (1613 registros)
✅ Generado: 09_Fontibon.xlsx (1781 registros)
✅ Generado: 18_Rafael Uribe Uribe.xlsx (2187 registros)
✅ Generado: 20_Sumapaz.xlsx (2579 registros)
✅ Generado: 10_Engativa.xlsx (3374 registros)
✅ Generado: 13_Teusaquillo.xlsx (1715 registros)
✅ Generado: 01_Usaquen.xlsx (2707 registros)
✅ Generado: 16_Puente Aranda.xlsx